# Chapter 4 — Nonparametric Filters: Particle Filter (MCL)

**Core formula:**
$$x_t^{[i]} \sim p(x_t | u_t, x_{t-1}^{[i]}), \quad w_t^{[i]} \propto p(z_t | x_t^{[i]}, m)$$

Each particle is a hypothesis. Each weight is that hypothesis's *right to survive* after seeing data.
When weights collapse → ESS drops → resample.

**Why EKF fails here:** symmetric environment creates a multimodal posterior.
Gaussian cannot represent two modes simultaneously.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))
import numpy as np
import matplotlib.pyplot as plt
from pluto_filters.pluto_filters.particle_filters.particle_filter import AugmentedMCL, Particle
from pluto_filters.pluto_filters.kalman_filters.ekf import LANDMARKS, normalize_angle

np.random.seed(7)
TRUE_POSE = np.array([1.0, 0.0, 0.0])

def true_measurement(landmark_id, pose=TRUE_POSE):
    lx, ly = LANDMARKS[landmark_id]
    dx, dy = lx - pose[0], ly - pose[1]
    r   = np.sqrt(dx**2 + dy**2) + np.random.normal(0, 0.2)
    phi = normalize_angle(np.arctan2(dy, dx) - pose[2]) + np.random.normal(0, 0.05)
    return r, phi

print("Imports OK")

In [ ]:
# ── Original MCL demo ───────────────────────────────────────────────────────
mcl = AugmentedMCL(n_particles=500, map_bounds=(-5, 5, -5, 5))
snapshots = []

def snapshot(label):
    xs = [p.x for p in mcl.particles]
    ys = [p.y for p in mcl.particles]
    ws = [p.weight for p in mcl.particles]
    snapshots.append((list(xs), list(ys), list(ws), label))

snapshot('Prior (uniform)')
r0, phi0 = true_measurement(0); mcl.update(0, r0, phi0); mcl.resample()
snapshot(f'After obs LM0  ESS={mcl.effective_n():.0f}')
mcl.predict(0.5, 0.0, 0.5); snapshot('After move +0.5m')
r2, phi2 = true_measurement(2); mcl.update(2, r2, phi2); mcl.resample()
snapshot(f'Converged  ESS={mcl.effective_n():.0f}')

x, y, theta = mcl.mean_pose()
print(f"Mean pose : ({x:.3f}, {y:.3f}, θ={np.degrees(theta):.1f}°)")
print(f"True pose : ({TRUE_POSE[0]:.3f}, {TRUE_POSE[1]:.3f}, 0.0°)")
print(f"Position error: {np.sqrt((x-TRUE_POSE[0])**2+(y-TRUE_POSE[1])**2):.3f} m")

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle('Particle Filter — Swarm of Plutos', fontsize=14)
for ax, (xs, ys, ws, label) in zip(axes.flat, snapshots):
    ws_arr = np.array(ws); ws_norm = ws_arr / (ws_arr.max() + 1e-300)
    ax.scatter(xs, ys, c=ws_norm, cmap='Blues', s=8, alpha=0.7, vmin=0, vmax=1)
    for lm_id, (lx, ly) in LANDMARKS.items(): ax.plot(lx, ly, 'r^', ms=8)
    ax.plot(*TRUE_POSE[:2], 'g*', ms=14, zorder=10, label='True')
    ax.set_xlim(-5.5, 5.5); ax.set_ylim(-5.5, 5.5)
    ax.set_title(label); ax.legend(fontsize=7); ax.grid(True, alpha=0.3); ax.set_aspect('equal')
plt.tight_layout(); plt.savefig('ch04_particle_filter.png', dpi=120); plt.show()

## Comparison 1 — Resampling Schemes: Systematic vs Multinomial vs Stratified

In [ ]:
# COMPARISON: resampling variance
def systematic_resample(particles, weights):
    N   = len(particles)
    pos = (np.arange(N) + np.random.uniform(0, 1)) / N
    cum = np.cumsum(weights); cum /= cum[-1]
    idx = np.searchsorted(cum, pos)
    return [particles[i] for i in idx]

def multinomial_resample(particles, weights):
    weights = np.array(weights); weights /= weights.sum()
    idx = np.random.choice(len(particles), len(particles), p=weights, replace=True)
    return [particles[i] for i in idx]

def stratified_resample(particles, weights):
    N   = len(particles)
    pos = (np.arange(N) + np.random.uniform(0, 1, N)) / N
    cum = np.cumsum(weights); cum /= cum[-1]
    idx = np.searchsorted(cum, pos)
    return [particles[i] for i in idx]

# Test with known distribution and measure variance of estimate
N_TRIALS = 500
N_P      = 200
true_val = 2.0
errors   = {m: [] for m in ['systematic', 'multinomial', 'stratified']}

for _ in range(N_TRIALS):
    xs_  = np.random.normal(true_val, 1.0, N_P)
    ws_  = np.random.dirichlet(np.ones(N_P))
    class FP: pass
    pts  = []; 
    for x_, w_ in zip(xs_, ws_): p=FP(); p.x=x_; p.weight=w_; pts.append(p)
    
    for method, fn in [('systematic', systematic_resample),
                        ('multinomial', multinomial_resample),
                        ('stratified',  stratified_resample)]:
        rs = fn(pts, ws_)
        est = np.mean([p.x for p in rs])
        errors[method].append((est - true_val)**2)

print("=== Resampling variance comparison (lower = better) ===")
for method, errs in errors.items():
    print(f"  {method:12s}  variance of estimate = {np.mean(errs):.6f}")
print("\nConclusion: systematic and stratified have lower variance than multinomial.")
print("Systematic is the most commonly used in practice.")

## Comparison 2 — Kidnapped Robot: Does Augmented MCL Recover?

In [ ]:
# KIDNAPPED ROBOT: teleport Pluto to a new location after convergence
np.random.seed(42)
mcl_kr = AugmentedMCL(n_particles=500, map_bounds=(-5, 5, -5, 5))

# Step 1: let filter converge at (1, 0, 0)
pose = np.array([1.0, 0.0, 0.0])
for lm in [0, 2, 3]:
    r_, p_ = true_measurement(lm, pose)
    mcl_kr.update(lm, r_, p_)
    mcl_kr.resample()
mcl_kr.predict(0.5, 0.0, 0.5)

pre_kidnap = [(p.x, p.y, p.weight) for p in mcl_kr.particles]
ess_before = mcl_kr.effective_n()

# Kidnap: teleport to (3, 3, 0)
KIDNAP_POSE = np.array([3.0, 3.0, 0.0])

# Step 2: observe from new location (filter doesn't know)
ess_log = [ess_before]
for step in range(15):
    for lm in [0, 2]:
        r_, p_ = true_measurement(lm, KIDNAP_POSE)
        mcl_kr.update(lm, r_, p_)
    mcl_kr.resample()
    mcl_kr.predict(0.3, 0.0, 0.5)
    ess_log.append(mcl_kr.effective_n())

post_kidnap = [(p.x, p.y, p.weight) for p in mcl_kr.particles]
x_est, y_est, _ = mcl_kr.mean_pose()
err_after = np.sqrt((x_est - KIDNAP_POSE[0])**2 + (y_est - KIDNAP_POSE[1])**2)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Kidnapped Robot — Augmented MCL Recovery', fontsize=13)

ax = axes[0]
xs_, ys_, ws_ = zip(*pre_kidnap)
ax.scatter(xs_, ys_, c=np.array(ws_)/max(ws_), cmap='Blues', s=8, alpha=0.7)
ax.plot(1, 0, 'g*', ms=14, label='True (before kidnap)'); ax.plot(3, 3, 'r*', ms=14, label='Kidnap target')
for lm_id, (lx, ly) in LANDMARKS.items(): ax.plot(lx, ly, 'k^', ms=7)
ax.set_title('Before kidnap (converged at (1,0))'); ax.set_xlim(-5,5); ax.set_ylim(-5,5)
ax.legend(fontsize=7); ax.set_aspect('equal')

ax = axes[1]
xs_, ys_, ws_ = zip(*post_kidnap)
ax.scatter(xs_, ys_, c=np.array(ws_)/max(ws_), cmap='Reds', s=8, alpha=0.7)
ax.plot(1, 0, 'g*', ms=14, label='Old true (kidnap source)'); ax.plot(3, 3, 'g^', ms=14, label='New true (kidnap target)')
ax.plot(x_est, y_est, 'b+', ms=14, mew=3, label=f'Estimate (err={err_after:.2f}m)')
for lm_id, (lx, ly) in LANDMARKS.items(): ax.plot(lx, ly, 'k^', ms=7)
ax.set_title('After 15 steps — recovering?'); ax.set_xlim(-5,5); ax.set_ylim(-5,5)
ax.legend(fontsize=7); ax.set_aspect('equal')

ax = axes[2]
ax.plot(ess_log, 'purple', lw=2, marker='o', ms=5)
ax.axvline(0, color='red', ls='--', label='Kidnap moment')
ax.axhline(0.5 * 500, color='orange', ls=':', label='ESS = N/2 threshold')
ax.set_xlabel('Step after kidnap'); ax.set_ylabel('Effective Sample Size (ESS)')
ax.set_title('ESS drops at kidnap → Augmented MCL injects random particles')
ax.legend(fontsize=8)

plt.tight_layout(); plt.savefig('ch04_kidnapped.png', dpi=120); plt.show()
print(f"ESS before kidnap: {ess_before:.1f}")
print(f"Error after recovery: {err_after:.3f} m")

## ✅ Compare | 🔥 Break | 📏 Measure

In [ ]:
# COMPARE: N_particles sensitivity
print("=== COMPARE: N_particles vs. error ===")
for N in [25, 100, 500, 2000]:
    m = AugmentedMCL(n_particles=N, map_bounds=(-5,5,-5,5))
    np.random.seed(42)
    for lm in [0, 2, 3]:
        r_, p_ = true_measurement(lm); m.update(lm, r_, p_)
    m.resample(); m.predict(0.5, 0.0, 0.5)
    r_, p_ = true_measurement(2); m.update(2, r_, p_); m.resample()
    xm, ym, _ = m.mean_pose()
    err = np.sqrt((xm-TRUE_POSE[0])**2+(ym-TRUE_POSE[1])**2)
    print(f"  N={N:5d}: error={err:.3f}m  ESS={m.effective_n():.1f}")

# BREAK: reduce to N=10 particles — particle deprivation
print("\n=== BREAK: N=10 (particle deprivation) ===")
m_tiny = AugmentedMCL(n_particles=10, map_bounds=(-5,5,-5,5))
for step in range(10):
    np.random.seed(step)
    r_, p_ = true_measurement(0); m_tiny.update(0, r_, p_); m_tiny.resample()
    xm, ym, _ = m_tiny.mean_pose()
    ess_ = m_tiny.effective_n()
    print(f"  step {step}: ESS={ess_:.1f}  particles left with weight>0: {sum(p.weight>1e-10 for p in m_tiny.particles)}")

# MEASURE
print("\n=== MEASURE: final metrics ===")
print(f"{'Metric':<35} {'Value':>10}")
print("-" * 48)
r0_, p0_ = true_measurement(0); r2_, p2_ = true_measurement(2)
mcl_final = AugmentedMCL(n_particles=500, map_bounds=(-5,5,-5,5))
for lm, (r_,p_) in [(0,(r0_,p0_)),(2,(r2_,p2_))]:
    mcl_final.update(lm,r_,p_); mcl_final.resample()
mcl_final.predict(0.5, 0.0, 0.5)
xf, yf, thf = mcl_final.mean_pose()
err_f = np.sqrt((xf-TRUE_POSE[0])**2+(yf-TRUE_POSE[1])**2)
print(f"{'Position error [m]':<35} {err_f:>10.4f}")
print(f"{'ESS (of N=500)':<35} {mcl_final.effective_n():>10.1f}")
print(f"{'ESS / N':<35} {mcl_final.effective_n()/500:>10.3f}")